# Análise das variáveis — Ativação de Sellers (Olist)

Notebook de **demonstração**: roda cada variável de `variaveis.md` contra o banco de validação local **`olist.db`** (SQLite) e mostra a tabela resultante (`seller_id` × colunas da feature).

Cada seção tem: a **explicação** (markdown) e o **SQL** (célula de código) — o mesmo SQL validado em `features_sqlite/`. Os scripts de produção em Spark estão em `features_spark/` (`workspace.olist.*`).

**Premissas-chave:** data de venda = `order_purchase_timestamp` (corte estrito `< {data_corte}`); **sem** filtro de `order_status`; grão = 1 linha por `seller_id`. Detalhes em [`docs/variaveis_detalhadas.md`](docs/variaveis_detalhadas.md).

👉 Para trocar a data de corte, edite `DATA_CORTE` na célula de setup e rode tudo.

In [ ]:
# === Setup: conexão, parâmetro de corte e helper ===
import os, sys, subprocess, sqlite3
import pandas as pd

DB = 'olist.db'
DATA_CORTE = '2018-07-01'   # <<< parâmetro de corte (altere aqui e rode tudo)

# gera o banco a partir de dados/ caso ainda não exista
if not os.path.exists(DB):
    subprocess.run([sys.executable, 'scripts/build_sqlite.py'], check=True)

con = sqlite3.connect(DB)

def run_sql(sql: str) -> pd.DataFrame:
    """Substitui {data_corte} pelo parâmetro e retorna o resultado como DataFrame."""
    return pd.read_sql_query(sql.replace('{data_corte}', DATA_CORTE), con)

print('Banco:', DB, '| data_corte =', DATA_CORTE)

## 1. `vlCategoriasDistintas`  `*`

Quantas **categorias diferentes** o seller vendeu em cada janela (`D28/D56/D365/Vida`). Categoria sem cadastro vira `'sem_categoria'`. Mede a **diversidade** do portfólio.

In [ ]:
# Feature 01_vlCategoriasDistintas  (fonte: features_sqlite/01_vlCategoriasDistintas.sql)
SQL = """
-- =====================================================================
-- vlCategoriasDistintas  (variaveis.md §1) — Diversidade de catálogo
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlCategoriasDistintas{D28,D56,D365,Vida}
-- Definição: nº de categorias (product_category_name) distintas dos
--            produtos VENDIDOS pelo seller em cada janela.
-- Grão     : a entidade é o SELLER; contamos categorias DISTINTAS, então
--            repetir o produto/pedido não infla o resultado.
-- Data     : order_purchase_timestamp, corte estrito < {data_corte}.
-- Premissas: sem filtro de order_status; categoria NULL -> 'sem_categoria'
--            (senão o COUNT(DISTINCT) a descartaria).
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: uma linha por item vendido até o corte (a base de tudo).
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders       o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'        -- Vida (corte estrito)
)
-- 2) Por janela, conta categorias distintas. O CASE restringe a janela e o
--    DISTINCT garante que cada categoria conte uma única vez.
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')
                        THEN categoria END) AS vlCategoriasDistintasD28,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')
                        THEN categoria END) AS vlCategoriasDistintasD56,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days')
                        THEN categoria END) AS vlCategoriasDistintasD365,
    COUNT(DISTINCT categoria)               AS vlCategoriasDistintasVida
FROM vendas
GROUP BY seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — existe categoria NULL? (justifica o COALESCE) -----------------------
-- Esperado: produtos_sem_categoria > 0 (há ~610 produtos sem categoria no dataset).
SELECT
    COUNT(*)                                                        AS itens_vendidos,
    SUM(CASE WHEN p.product_category_name IS NULL THEN 1 ELSE 0 END) AS itens_sem_categoria,
    COUNT(DISTINCT CASE WHEN p.product_category_name IS NULL
                        THEN oi.product_id END)                     AS produtos_sem_categoria
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
LEFT JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}';

-- ----------------------- Prova B — monotonia D28 <= D56 <= D365 <= Vida -----------------------
-- Como D28 ⊂ D56 ⊂ D365 ⊂ Vida, a contagem só pode crescer. Esperado: 0 violações.
WITH vendas AS (
    SELECT oi.seller_id,
           COALESCE(p.product_category_name,'sem_categoria') AS categoria,
           o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
por_seller AS (
    SELECT seller_id,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN categoria END) AS d28,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN categoria END) AS d56,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN categoria END) AS d365,
        COUNT(DISTINCT categoria) AS vida
    FROM vendas GROUP BY seller_id
)
SELECT COUNT(*) AS violacoes_monotonia
FROM por_seller
WHERE NOT (d28 <= d56 AND d56 <= d365 AND d365 <= vida);
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 2. `vlProdutosDistintos`  `*`

Quantos **`product_id` diferentes** o seller vendeu por janela. Amplitude do catálogo efetivamente vendido.

In [ ]:
# Feature 02_vlProdutosDistintos  (fonte: features_sqlite/02_vlProdutosDistintos.sql)
SQL = """
-- =====================================================================
-- vlProdutosDistintos  (variaveis.md §1) — Diversidade de catálogo
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlProdutosDistintos{D28,D56,D365,Vida}
-- Definição: nº de product_id (SKU) distintos vendidos pelo seller em cada
--            janela.
-- Grão     : entidade = SELLER; contamos SKUs DISTINTOS, logo o mesmo
--            produto vendido em vários pedidos conta uma vez só.
-- Data     : order_purchase_timestamp, corte estrito < {data_corte}.
-- Premissas: "produto" = product_id; sem filtro de order_status.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: uma linha por item vendido até o corte.
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
-- 2) Por janela, conta SKUs distintos (CASE restringe a janela; DISTINCT dedup).
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')
                        THEN product_id END) AS vlProdutosDistintosD28,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')
                        THEN product_id END) AS vlProdutosDistintosD56,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days')
                        THEN product_id END) AS vlProdutosDistintosD365,
    COUNT(DISTINCT product_id)               AS vlProdutosDistintosVida
FROM vendas
GROUP BY seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — DISTINCT importa: SKU vendido em vários pedidos conta 1 -----------------------
-- Mostra SKUs com mais de uma venda (sem DISTINCT, eles inflariam a contagem).
SELECT oi.seller_id, oi.product_id, COUNT(*) AS vezes_vendido
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id, oi.product_id
HAVING COUNT(*) > 1
ORDER BY vezes_vendido DESC
LIMIT 20;

-- ----------------------- Prova B — monotonia D28 <= D56 <= D365 <= Vida -----------------------
-- Esperado: 0 violações.
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id, o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
por_seller AS (
    SELECT seller_id,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN product_id END) AS d28,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN product_id END) AS d56,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN product_id END) AS d365,
        COUNT(DISTINCT product_id) AS vida
    FROM vendas GROUP BY seller_id
)
SELECT COUNT(*) AS violacoes_monotonia
FROM por_seller
WHERE NOT (d28 <= d56 AND d56 <= d365 AND d365 <= vida);
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 3. `vlContagemCategoriaConcorrentes`  `*`

Para cada seller, quantos **outros** sellers venderam em **alguma categoria em comum**, na mesma janela. Concorrência **indireta**.

In [ ]:
# Feature 03_vlContagemCategoriaConcorrentes  (fonte: features_sqlite/03_vlContagemCategoriaConcorrentes.sql)
SQL = """
-- =====================================================================
-- vlContagemCategoriaConcorrentes  (variaveis.md §2) — Concorrência indireta
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlContagemCategoriaConcorrentes{D28,D56,D365,Vida}
-- Definição: para o seller A, nº de OUTROS sellers (B<>A) que venderam em
--            ALGUMA categoria em comum com A na mesma janela (substitutos).
-- Grão     : entidade = SELLER A; COUNT(DISTINCT concorrente) impede contar
--            o mesmo B duas vezes quando ele divide várias categorias com A.
-- Data     : order_purchase_timestamp, corte estrito < {data_corte}.
-- Premissas: sem filtro de order_status; categoria NULL -> 'sem_categoria';
--            janela sem venda -> 0 concorrentes (não está no mercado).
-- Modelagem: em vez de self-join (sc a JOIN sc b), nomeamos os dois papéis:
--            "minhas_cat" (categorias do seller) x "roster" (todos os sellers
--            por categoria). O cruzamento fica explícito e legível.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens vendidos até o corte, com a categoria e a data.
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders       o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) pares distintos (seller, categoria, janela): "quem atua em qual categoria".
--    Uma flag por janela evita repetir a base 4 vezes.
atuacao AS (
    SELECT DISTINCT
        seller_id,
        categoria,
        CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN 1 ELSE 0 END AS in_d28,
        CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN 1 ELSE 0 END AS in_d56,
        CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN 1 ELSE 0 END AS in_d365
    FROM vendas
),
-- 3) colapsa para 1 linha por (seller, categoria) carregando a janela mais curta
--    em que o par aparece (MAX das flags).
minhas_cat AS (
    SELECT seller_id, categoria,
           MAX(in_d28) AS in_d28, MAX(in_d56) AS in_d56, MAX(in_d365) AS in_d365
    FROM atuacao
    GROUP BY seller_id, categoria
),
-- 4) cruza A (minhas_cat) com B (mesma tabela como "roster") na MESMA categoria,
--    exigindo que ambos estejam ativos na janela e que B <> A.
concorrentes AS (
    SELECT
        a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS q_d28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS q_d56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS q_d365,
        COUNT(DISTINCT b.seller_id)                                                    AS q_vida
    FROM minhas_cat a
    JOIN minhas_cat b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
-- 5) spine + COALESCE 0: seller sem concorrente em nenhuma janela some do JOIN.
SELECT
    s.seller_id,
    COALESCE(c.q_d28,  0) AS vlContagemCategoriaConcorrentesD28,
    COALESCE(c.q_d56,  0) AS vlContagemCategoriaConcorrentesD56,
    COALESCE(c.q_d365, 0) AS vlContagemCategoriaConcorrentesD365,
    COALESCE(c.q_vida, 0) AS vlContagemCategoriaConcorrentesVida
FROM spine s
LEFT JOIN concorrentes c ON c.seller_id = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — por que COUNT(DISTINCT concorrente): pares que dividem >1 categoria (Vida) -----------------------
-- Sem DISTINCT, o concorrente B seria contado uma vez por categoria em comum.
WITH minhas_categorias AS (
    SELECT DISTINCT oi.seller_id,
           COALESCE(p.product_category_name,'sem_categoria') AS categoria
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT a.seller_id AS seller_a, b.seller_id AS concorrente_b,
       COUNT(*) AS categorias_em_comum
FROM minhas_categorias a
JOIN minhas_categorias b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id
GROUP BY a.seller_id, b.seller_id
HAVING COUNT(*) > 1
ORDER BY categorias_em_comum DESC
LIMIT 20;

-- ----------------------- Prova B — categorias com um único seller geram 0 concorrentes (-> spine + COALESCE) -----------------------
WITH minhas_categorias AS (
    SELECT DISTINCT oi.seller_id,
           COALESCE(p.product_category_name,'sem_categoria') AS categoria
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT categoria, COUNT(DISTINCT seller_id) AS sellers_na_categoria
FROM minhas_categorias
GROUP BY categoria
HAVING COUNT(DISTINCT seller_id) = 1
ORDER BY categoria
LIMIT 20;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 4. `vlContagemProdutosConcorrentes`  `*`

Quantos **outros** sellers venderam o **mesmo `product_id`**, por janela. Concorrência **direta** (mesmo SKU → guerra de preço).

In [ ]:
# Feature 04_vlContagemProdutosConcorrentes  (fonte: features_sqlite/04_vlContagemProdutosConcorrentes.sql)
SQL = """
-- =====================================================================
-- vlContagemProdutosConcorrentes  (variaveis.md §2) — Concorrência direta
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlContagemProdutosConcorrentes{D28,D56,D365,Vida}
-- Definição: para o seller A, nº de OUTROS sellers (B<>A) que venderam o
--            MESMO product_id que A na mesma janela (concorrência DIRETA —
--            o cliente poderia comprar o SKU idêntico de outro seller).
-- Grão     : entidade = SELLER A; COUNT(DISTINCT concorrente) impede contar
--            o mesmo B duas vezes quando ele divide vários SKUs com A.
-- Data     : order_purchase_timestamp, corte estrito < {data_corte}.
-- Premissas: sem filtro de order_status; 1.225 product_id são vendidos por
--            >1 seller no dataset; janela sem venda -> 0.
-- Modelagem: mesmo padrão da §3, trocando categoria por product_id.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- ---------------------------------------------------------------------
-- LIMITAÇÃO CONHECIDA (interpretação): esta métrica é um PISO de concorrência
--   — alta precisão, baixo recall. `product_id` é o SKU ÚNICO do catálogo Olist
--   (chave de `products`, reusado em várias vendas: 32.951 produtos x 112.650
--   itens, média 3,4 vendas/produto — NÃO é gerado por venda). Só 1.225 produtos
--   (~3,7%) são vendidos por >1 seller (modelo de marketplace/buy box: a mesma
--   página de produto disputada por vários sellers — estruturalmente válido).
--   A variável SÓ enxerga concorrência onde o catálogo JÁ unificou o SKU. Ela é
--   CEGA a produtos FISICAMENTE EQUIVALENTES cadastrados com product_id distintos
--   (ex.: o mesmo isqueiro anunciado 2x). O dataset não tem marca/EAN/GTIN/título
--   para casar equivalentes, então essa concorrência fica invisível. Portanto:
--   um 0 NÃO garante ausência de concorrência. Use em par com a §3 (concorrência
--   por categoria = "teto" largo). Prova C abaixo materializa o piso.
-- =====================================================================

-- 1) vendas: itens vendidos até o corte, com produto e data.
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) pares distintos (seller, produto) com flags de janela.
atuacao AS (
    SELECT DISTINCT
        seller_id,
        product_id,
        CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN 1 ELSE 0 END AS in_d28,
        CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN 1 ELSE 0 END AS in_d56,
        CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN 1 ELSE 0 END AS in_d365
    FROM vendas
),
meus_produtos AS (
    SELECT seller_id, product_id,
           MAX(in_d28) AS in_d28, MAX(in_d56) AS in_d56, MAX(in_d365) AS in_d365
    FROM atuacao
    GROUP BY seller_id, product_id
),
-- 3) cruza A com B no MESMO product_id, ambos ativos na janela, B <> A.
concorrentes AS (
    SELECT
        a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS q_d28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS q_d56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS q_d365,
        COUNT(DISTINCT b.seller_id)                                                    AS q_vida
    FROM meus_produtos a
    JOIN meus_produtos b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT
    s.seller_id,
    COALESCE(c.q_d28,  0) AS vlContagemProdutosConcorrentesD28,
    COALESCE(c.q_d56,  0) AS vlContagemProdutosConcorrentesD56,
    COALESCE(c.q_d365, 0) AS vlContagemProdutosConcorrentesD365,
    COALESCE(c.q_vida, 0) AS vlContagemProdutosConcorrentesVida
FROM spine s
LEFT JOIN concorrentes c ON c.seller_id = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — SKUs vendidos por >1 seller (a métrica só faz sentido por causa deles) -----------------------
-- Esperado: ~1.225 product_id disputados.
WITH produtos_disputados AS (
    SELECT oi.product_id, COUNT(DISTINCT oi.seller_id) AS sellers_no_sku
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
    GROUP BY oi.product_id
    HAVING COUNT(DISTINCT oi.seller_id) > 1
)
SELECT COUNT(*) AS skus_disputados, MAX(sellers_no_sku) AS max_sellers_num_sku
FROM produtos_disputados;

-- ----------------------- Prova B — por que COUNT(DISTINCT concorrente): B que divide >1 SKU com A (Vida) -----------------------
WITH meus_produtos AS (
    SELECT DISTINCT oi.seller_id, oi.product_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT a.seller_id AS seller_a, b.seller_id AS concorrente_b,
       COUNT(*) AS skus_em_comum
FROM meus_produtos a
JOIN meus_produtos b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id
GROUP BY a.seller_id, b.seller_id
HAVING COUNT(*) > 1
ORDER BY skus_em_comum DESC
LIMIT 20;

-- ----------------------- Prova C — dois sellers DISTINTOS no MESMO product_id, em VENDAS DISTINTAS -----------------------
-- Comprova a semântica da concorrência: o mesmo SKU do catálogo é vendido por
-- A e por B em PEDIDOS diferentes (não é venda casada no mesmo carrinho).
--   `b.seller_id <> a.seller_id`  -> sellers diferentes (concorrentes).
--   `b.order_id  <> a.order_id`   -> vendas (pedidos) diferentes.
--   `a.seller_id <  b.seller_id`  -> 1 linha por par (evita o espelho B,A).
-- Esperado: ~980 product_id disputados em vendas separadas (cutoff 2018-07-01).
WITH vendas AS (
    SELECT oi.product_id, oi.seller_id, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT a.product_id,
       a.seller_id AS seller_a, a.order_id AS pedido_a,
       b.seller_id AS seller_b, b.order_id AS pedido_b
FROM vendas a
JOIN vendas b
  ON b.product_id = a.product_id
 AND b.seller_id <> a.seller_id
 AND b.order_id  <> a.order_id
 AND a.seller_id <  b.seller_id
ORDER BY a.product_id
LIMIT 20;

-- ----------------------- Prova D — contra-prova: concorrência NUNCA é co-ocorrência no mesmo pedido -----------------------
-- Se A e B vendem o mesmo product_id, NUNCA é dentro do mesmo order_id (carrinho).
-- Garante que a Prova C mede concorrência (vendas separadas), não venda casada.
-- Esperado: 0 (cutoff 2018-07-01).
WITH vendas AS (
    SELECT oi.product_id, oi.seller_id, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT COUNT(*) AS pares_mesmo_pedido_mesmo_produto_sellers_distintos
FROM vendas a
JOIN vendas b
  ON b.product_id = a.product_id
 AND b.order_id  = a.order_id
 AND b.seller_id <> a.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 5. `vlCaracteresDescricao`  (estático — Vida)

Média, p25, p50/mediana, p75, min e max do tamanho da descrição dos **produtos distintos** vendidos (cada SKU pesa 1). Qualidade do cadastro. **NULL → 0**: produto sem descrição conta como 0 caractere (puxa média/min).

In [ ]:
# Feature 05_vlCaracteresDescricao  (fonte: features_sqlite/05_vlCaracteresDescricao.sql)
SQL = """
-- =====================================================================
-- vlCaracteresDescricao  (variaveis.md §3) — Atributo de cadastro (estático)
-- Janela: Vida (sem sufixo)        |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlMediaCaracteresDescricao, vlMedianaCaracteresDescricao,
--            vl25CaracteresDescricao, vl50CaracteresDescricao,
--            vl75CaracteresDescricao, vlMinCaracteresDescricao,
--            vlMaxCaracteresDescricao.
-- Definição: estatísticas de product_description_lenght.
-- ---------------------------------------------------------------------
-- GRÃO (importante): descrição é atributo do PRODUTO (cadastro), não da
--   venda. A entidade aqui é o PRODUTO. Por isso tomamos os PRODUTOS
--   DISTINTOS vendidos por cada seller (DISTINCT product_id) — assim o
--   mesmo SKU vendido em vários pedidos NÃO entra repetido e não enviesa a
--   média/percentis. (vlMediana == vl50: a mediana é o percentil 50.)
-- Data     : order_purchase_timestamp < {data_corte}.
-- Percentil: SQLite não tem percentile(); usamos o percentil CONTÍNUO
--   tipo-7 (interpolação linear, idêntico ao percentile() do Spark):
--   idx = (n-1)*p; valor = v[floor(idx)] + frac*(v[floor(idx)+1]-v[floor(idx)]).
-- NULOS (decisão do cliente): descrição NULL é tratada como ZERO caractere,
--   NÃO ignorada. Quantitativamente é equivalente — o produto SEM descrição
--   continua na base e PUXA a média/mínimo para baixo (sinaliza catálogo mal
--   cadastrado, que ajuda a prever inativação). COALESCE(L,0). Não há valor 0
--   "natural" na base (descrição vazia aparece como NULL), então o 0 é nosso.
--   Seller sem NENHUM produto (impossível aqui, todos têm ≥1 venda) -> NULL.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens vendidos até o corte (define o universo de sellers).
--    COALESCE(L,0): descrição ausente conta como 0 caractere (não é descartada).
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id, COALESCE(p.product_description_lenght, 0) AS L
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) produtos: PRODUTOS DISTINTOS por seller (cada SKU pesa 1). TODOS entram,
--    inclusive os de descrição 0 (antes ausente) — daí não há filtro de NULL.
produtos AS (
    SELECT DISTINCT seller_id, product_id, L
    FROM vendas
),
-- 3) ranqueia L dentro de cada seller para o percentil tipo-7.
ranked AS (
    SELECT seller_id, L,
           CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY L) AS INTEGER) - 1 AS rn0,
           COUNT(*) OVER (PARTITION BY seller_id)                                     AS n
    FROM produtos
),
-- 4) componentes do percentil (limite inferior, superior e fração) por seller.
agg AS (
    SELECT
        seller_id, n,
        AVG(L) AS media, MIN(L) AS mn, MAX(L) AS mx,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER)     THEN L END) AS p25_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER) + 1 THEN L END) AS p25_hi,
        (n-1)*0.25 - CAST((n-1)*0.25 AS INTEGER)                        AS p25_f,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER)     THEN L END) AS p50_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER) + 1 THEN L END) AS p50_hi,
        (n-1)*0.50 - CAST((n-1)*0.50 AS INTEGER)                        AS p50_f,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER)     THEN L END) AS p75_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER) + 1 THEN L END) AS p75_hi,
        (n-1)*0.75 - CAST((n-1)*0.75 AS INTEGER)                        AS p75_f
    FROM ranked
    GROUP BY seller_id, n
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
-- 5) interpola cada percentil; vlMediana e vl50 são o mesmo valor (p50).
SELECT
    s.seller_id,
    agg.media                                                                                              AS vlMediaCaracteresDescricao,
    CASE WHEN agg.p50_hi IS NULL THEN agg.p50_lo ELSE agg.p50_lo + agg.p50_f*(agg.p50_hi-agg.p50_lo) END AS vlMedianaCaracteresDescricao,
    CASE WHEN agg.p25_hi IS NULL THEN agg.p25_lo ELSE agg.p25_lo + agg.p25_f*(agg.p25_hi-agg.p25_lo) END AS vl25CaracteresDescricao,
    CASE WHEN agg.p50_hi IS NULL THEN agg.p50_lo ELSE agg.p50_lo + agg.p50_f*(agg.p50_hi-agg.p50_lo) END AS vl50CaracteresDescricao,
    CASE WHEN agg.p75_hi IS NULL THEN agg.p75_lo ELSE agg.p75_lo + agg.p75_f*(agg.p75_hi-agg.p75_lo) END AS vl75CaracteresDescricao,
    agg.mn                                                                                                 AS vlMinCaracteresDescricao,
    agg.mx                                                                                                 AS vlMaxCaracteresDescricao
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — DISTINCT produto: o mesmo SKU vendido N vezes NÃO infla a estatística -----------------------
-- Lista SKUs vendidos em vários pedidos; sem o DISTINCT, a descrição deles
-- entraria N vezes na média/percentis (viés pela entidade errada — venda, não produto).
SELECT oi.seller_id, oi.product_id,
       p.product_description_lenght AS descricao_chars,
       COUNT(*) AS unidades_vendidas
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id, oi.product_id, p.product_description_lenght
HAVING COUNT(*) > 1
ORDER BY unidades_vendidas DESC
LIMIT 20;

-- ----------------------- Prova B — descrição NULL conta como 0 (decisão do cliente): quantos produtos viram 0? -----------------------
-- Esperado: ~580 produtos distintos (<corte) sem descrição entram com L=0,
-- afetando média e mínimo (o mínimo de muitos sellers cai para 0).
WITH produtos_distintos AS (
    SELECT DISTINCT oi.product_id, p.product_description_lenght AS L
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT COUNT(*) AS produtos_distintos_vendidos,
       SUM(CASE WHEN L IS NULL THEN 1 ELSE 0 END) AS produtos_que_viram_zero
FROM produtos_distintos;

-- ----------------------- Prova C — efeito do COALESCE na média: comparação ignorar-NULL vs NULL=0 -----------------------
-- Mostra, por seller, a média descartando NULL (antiga) vs tratando NULL como 0
-- (atual). Diferem exatamente nos sellers que vendem algum produto sem descrição.
WITH produtos AS (
    SELECT DISTINCT oi.seller_id, oi.product_id, p.product_description_lenght AS L
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT seller_id,
       AVG(L)              AS media_ignorando_null,
       AVG(COALESCE(L, 0)) AS media_null_zero,
       SUM(CASE WHEN L IS NULL THEN 1 ELSE 0 END) AS produtos_sem_descricao
FROM produtos
GROUP BY seller_id
HAVING produtos_sem_descricao > 0
ORDER BY produtos_sem_descricao DESC
LIMIT 20;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 6. `vlMediaFotosProduto`  (estático — Vida)

Média de `product_photos_qty` entre os **produtos distintos** vendidos. Proxy de qualidade da vitrine. **NULL → 0**: produto sem foto conta como 0 (não há 0 'natural' na base; o mínimo real é 1).

In [ ]:
# Feature 06_vlMediaFotosProduto  (fonte: features_sqlite/06_vlMediaFotosProduto.sql)
SQL = """
-- =====================================================================
-- vlMediaFotosProduto  (variaveis.md §3) — Atributo de cadastro (estático)
-- Janela: Vida (sem sufixo)        |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Coluna   : vlMediaFotosProduto
-- Definição: média de product_photos_qty por produto do seller.
-- ---------------------------------------------------------------------
-- GRÃO (importante): nº de fotos é atributo do PRODUTO (cadastro). A
--   entidade é o PRODUTO, então tomamos os PRODUTOS DISTINTOS vendidos por
--   cada seller (DISTINCT product_id) — o mesmo SKU vendido várias vezes
--   pesa 1 e não enviesa a média.
-- Data     : order_purchase_timestamp < {data_corte}.
-- NULOS (decisão do cliente): fotos NULL conta como ZERO foto
--   (COALESCE(fotos,0)), NÃO é ignorada. No dataset não existe foto=0
--   "natural" (mínimo real = 1); os ~580 produtos sem foto aparecem como
--   NULL, e tratá-los como 0 mantém o produto na base e PUXA a média/mínimo
--   para baixo — "não cadastrou foto" é informação (catálogo fraco, pior
--   conversão), que ajuda a prever inativação.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens vendidos até o corte (universo de sellers).
--    COALESCE(fotos,0): produto sem foto conta como 0 (não é descartado).
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id, COALESCE(p.product_photos_qty, 0) AS fotos
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) produtos: PRODUTOS DISTINTOS por seller (cada SKU pesa 1). TODOS entram.
produtos AS (
    SELECT DISTINCT seller_id, product_id, fotos
    FROM vendas
),
-- 3) média de fotos por produto distinto (0 incluído).
agg AS (
    SELECT seller_id, AVG(fotos) AS vlMediaFotosProduto
    FROM produtos
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT s.seller_id, agg.vlMediaFotosProduto
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — DISTINCT produto: SKU vendido N vezes pesa 1 na média -----------------------
SELECT oi.seller_id, oi.product_id,
       p.product_photos_qty AS fotos,
       COUNT(*) AS unidades_vendidas
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id, oi.product_id, p.product_photos_qty
HAVING COUNT(*) > 1
ORDER BY unidades_vendidas DESC
LIMIT 20;

-- ----------------------- Prova B — fotos NULL conta como 0 (decisão do cliente): não há 0 "natural"; quantos viram 0? -----------------------
-- min_fotos_real é o menor valor NÃO-nulo (esperado 1): confirma que o 0 só
-- existe porque criamos (NULL->0). produtos_que_viram_zero ~580 (<corte).
WITH produtos_distintos AS (
    SELECT DISTINCT oi.product_id, p.product_photos_qty AS fotos
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT COUNT(*) AS produtos_distintos_vendidos,
       SUM(CASE WHEN fotos IS NULL THEN 1 ELSE 0 END) AS produtos_que_viram_zero,
       MIN(fotos) AS min_fotos_real
FROM produtos_distintos;

-- ----------------------- Prova C — efeito do COALESCE na média: ignorar-NULL vs NULL=0 -----------------------
WITH produtos AS (
    SELECT DISTINCT oi.seller_id, oi.product_id, p.product_photos_qty AS fotos
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT seller_id,
       AVG(fotos)              AS media_ignorando_null,
       AVG(COALESCE(fotos, 0)) AS media_null_zero,
       SUM(CASE WHEN fotos IS NULL THEN 1 ELSE 0 END) AS produtos_sem_foto
FROM produtos
GROUP BY seller_id
HAVING produtos_sem_foto > 0
ORDER BY produtos_sem_foto DESC
LIMIT 20;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 7. `vlPesoProduto`  (distribuição estática + total `*`)

Distribuição (média/mediana/p25/p50/p75/min/max) de `product_weight_g` (g) sobre **produtos distintos** — **estática** (peso é atributo imutável do produto, não muda no tempo) + **`vlTotalPesoProdutos{D28,D56,D365,Vida}`** (kg) somado por **unidade** embarcada (massa cresce com vendas → 4 janelas).

In [ ]:
# Feature 07_vlPesoProduto  (fonte: features_sqlite/07_vlPesoProduto.sql)
SQL = """
-- =====================================================================
-- vlPesoProduto  (variaveis.md §4) — Peso dos produtos
-- Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas:
--   DISTRIBUIÇÃO (ESTÁTICA, sem sufixo — peso é atributo do produto):
--     vl{Media,Mediana,25,50,75,Min,Max}PesoProduto   -> em GRAMAS
--   TOTAL (4 janelas D28/D56/D365/Vida — massa embarcada muda no tempo):
--     vlTotalPesoProdutos{W}                           -> em KG
-- ---------------------------------------------------------------------
-- DECISÃO DO CLIENTE (atributo de produto não varia no tempo):
--   Peso vem da tabela `products` (1 linha por product_id), é IMUTÁVEL — não
--   muda entre D28/D56/D365/Vida. Logo a DISTRIBUIÇÃO (media/mediana/percentis/
--   min/max) é ESTÁTICA: calculada UMA vez sobre os PRODUTOS DISTINTOS que o
--   seller já vendeu (toda a vida < corte). Antes havia 4 janelas redundantes
--   (mesmo atributo, só mudava quais SKUs entravam) — eliminadas, alinhando
--   com descrição/fotos (§3). (vlMediana == vl50: a mediana é o percentil 50.)
--
--   O TOTAL (vlTotalPesoProdutos) é EXCEÇÃO e MANTÉM as 4 janelas: é a MASSA
--   EMBARCADA, soma por UNIDADE vendida (cada linha de order_items), e portanto
--   CRESCE com novas vendas. Mesma lógica que justifica janelas nos R$/kg (§6).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Percentil: SQLite não tem percentile(); percentil contínuo tipo-7
--   (interpolação linear), idêntico ao percentile() do Spark.
-- Nulos    : peso NULL ignorado (2 produtos sem peso); seller sem produto com
--            peso -> distribuição NULL; janela sem venda -> total NULL.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: uma linha por unidade vendida até o corte.
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id,
           p.product_weight_g         AS w,
           o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) TOTAL por UNIDADE (massa embarcada, g->kg), com as 4 janelas. Cada venda
--    soma uma vez — total cresce no tempo, por isso mantém D28/D56/D365/Vida.
total AS (
    SELECT seller_id,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN w END)/1000.0 AS tot_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN w END)/1000.0 AS tot_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN w END)/1000.0 AS tot_d365,
        SUM(w)/1000.0                                                                      AS tot_vida
    FROM vendas
    GROUP BY seller_id
),
-- 3) PRODUTOS DISTINTOS por seller (toda a vida; peso constante por produto).
--    Base ESTÁTICA da distribuição — sem janelas, pois o peso não muda.
produtos AS (
    SELECT DISTINCT seller_id, product_id, w
    FROM vendas
    WHERE w IS NOT NULL
),
-- 4) ranqueia o peso dentro de cada seller para o percentil tipo-7.
ranked AS (
    SELECT seller_id, w,
           CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY w) AS INTEGER) - 1 AS rn0,
           COUNT(*) OVER (PARTITION BY seller_id)                                     AS n
    FROM produtos
),
-- 5) componentes do percentil (limite inferior, superior e fração) por seller.
agg AS (
    SELECT
        seller_id, n,
        AVG(w) AS media, MIN(w) AS mn, MAX(w) AS mx,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER)     THEN w END) AS p25_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER) + 1 THEN w END) AS p25_hi,
        (n-1)*0.25 - CAST((n-1)*0.25 AS INTEGER)                        AS p25_f,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER)     THEN w END) AS p50_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER) + 1 THEN w END) AS p50_hi,
        (n-1)*0.50 - CAST((n-1)*0.50 AS INTEGER)                        AS p50_f,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER)     THEN w END) AS p75_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER) + 1 THEN w END) AS p75_hi,
        (n-1)*0.75 - CAST((n-1)*0.75 AS INTEGER)                        AS p75_f
    FROM ranked
    GROUP BY seller_id, n
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
-- 6) interpola os percentis (distribuição estática) e anexa os totais (4 janelas).
SELECT
    s.seller_id,
    agg.media                                                                                          AS vlMediaPesoProduto,
    CASE WHEN agg.p50_hi IS NULL THEN agg.p50_lo ELSE agg.p50_lo + agg.p50_f*(agg.p50_hi-agg.p50_lo) END AS vlMedianaPesoProduto,
    CASE WHEN agg.p25_hi IS NULL THEN agg.p25_lo ELSE agg.p25_lo + agg.p25_f*(agg.p25_hi-agg.p25_lo) END AS vl25PesoProduto,
    CASE WHEN agg.p50_hi IS NULL THEN agg.p50_lo ELSE agg.p50_lo + agg.p50_f*(agg.p50_hi-agg.p50_lo) END AS vl50PesoProduto,
    CASE WHEN agg.p75_hi IS NULL THEN agg.p75_lo ELSE agg.p75_lo + agg.p75_f*(agg.p75_hi-agg.p75_lo) END AS vl75PesoProduto,
    agg.mn                                                                                              AS vlMinPesoProduto,
    agg.mx                                                                                              AS vlMaxPesoProduto,
    t.tot_d28  AS vlTotalPesoProdutosD28,
    t.tot_d56  AS vlTotalPesoProdutosD56,
    t.tot_d365 AS vlTotalPesoProdutosD365,
    t.tot_vida AS vlTotalPesoProdutosVida
FROM spine s
LEFT JOIN agg   ON agg.seller_id = s.seller_id
LEFT JOIN total t ON t.seller_id  = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — peso é ESTÁTICO: D28/D56/D365/Vida dariam o MESMO valor por produto -----------------------
-- O peso é o mesmo em qualquer janela; a janela só mudaria QUAIS SKUs entram,
-- não o valor de cada um. Por isso a distribuição não precisa de sufixo de período.
SELECT oi.product_id,
       COUNT(DISTINCT p.product_weight_g) AS valores_distintos_de_peso
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY oi.product_id
HAVING valores_distintos_de_peso > 1   -- esperado: 0 linhas (peso nunca muda por produto)
LIMIT 20;

-- ----------------------- Prova B — DISTRIBUIÇÃO usa produto distinto; TOTAL usa unidade (grãos diferentes) -----------------------
-- Para um SKU vendido N vezes: na distribuição entra 1x; no total entra N vezes.
SELECT oi.seller_id, oi.product_id, p.product_weight_g AS peso_g,
       COUNT(*) AS unidades_vendidas
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id, oi.product_id, p.product_weight_g
HAVING COUNT(*) > 1
ORDER BY unidades_vendidas DESC
LIMIT 20;

-- ----------------------- Prova C — há peso NULL? (ignorado na distribuição e no total) -----------------------
-- Esperado: produtos_sem_peso = 2.
SELECT COUNT(*) AS itens_vendidos,
       SUM(CASE WHEN p.product_weight_g IS NULL THEN 1 ELSE 0 END) AS itens_sem_peso,
       COUNT(DISTINCT CASE WHEN p.product_weight_g IS NULL THEN oi.product_id END) AS produtos_sem_peso
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}';
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 8. `vlCubagemProdutos`  (média estática + total `*`)

**Média** da cubagem (cm³ = `L×H×W`) sobre produtos distintos — **estática** (atributo imutável) + **`vlTotalCubagemProdutos{D28,D56,D365,Vida}`** somado por unidade embarcada (volume cresce com vendas → 4 janelas).

In [ ]:
# Feature 08_vlCubagemProdutos  (fonte: features_sqlite/08_vlCubagemProdutos.sql)
SQL = """
-- =====================================================================
-- vlCubagemProdutos  (variaveis.md §5) — Cubagem (volume da caixa de envio)
-- Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas:
--   MÉDIA (ESTÁTICA, sem sufixo — cubagem é atributo do produto):
--     vlMediaCubagemProdutos                           -> em cm³
--   TOTAL (4 janelas D28/D56/D365/Vida — volume embarcado muda no tempo):
--     vlTotalCubagemProdutos{W}                        -> em cm³
-- ---------------------------------------------------------------------
-- DECISÃO DO CLIENTE (atributo de produto não varia no tempo):
--   cubagem = product_length_cm × product_height_cm × product_width_cm, todas
--   vindas de `products` (1 linha por product_id) — IMUTÁVEL. A MÉDIA é
--   ESTÁTICA: calculada UMA vez sobre os PRODUTOS DISTINTOS já vendidos pelo
--   seller (toda a vida < corte), alinhando com descrição/fotos (§3). Antes
--   havia 4 janelas redundantes (mesmo atributo) — eliminadas.
--   O TOTAL é EXCEÇÃO e mantém 4 janelas: é o VOLUME EMBARCADO (soma por
--   UNIDADE vendida), cresce com novas vendas — mesma lógica dos R$/kg (§6).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Nulos    : qualquer dimensão NULL -> cubagem NULL (ignorada na média e no
--            total); seller sem produto com cubagem -> média NULL; janela sem
--            venda -> total NULL.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: uma linha por unidade vendida, com a cubagem do produto.
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id,
           (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cub,
           o.order_purchase_timestamp                                       AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) TOTAL por UNIDADE (volume embarcado), 4 janelas. Cada venda soma uma vez
--    — total cresce no tempo, por isso mantém D28/D56/D365/Vida.
total AS (
    SELECT seller_id,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN cub END) AS tot_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN cub END) AS tot_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN cub END) AS tot_d365,
        SUM(cub)                                                                      AS tot_vida
    FROM vendas
    GROUP BY seller_id
),
-- 3) PRODUTOS DISTINTOS por seller (toda a vida; cub constante por produto).
--    Base ESTÁTICA da média — sem janelas, pois a cubagem não muda.
produtos AS (
    SELECT DISTINCT seller_id, product_id, cub
    FROM vendas
    WHERE cub IS NOT NULL
),
-- 4) média estática da cubagem sobre os produtos distintos.
media AS (
    SELECT seller_id, AVG(cub) AS med_vida
    FROM produtos
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT
    s.seller_id,
    m.med_vida AS vlMediaCubagemProdutos,
    t.tot_d28  AS vlTotalCubagemProdutosD28,
    t.tot_d56  AS vlTotalCubagemProdutosD56,
    t.tot_d365 AS vlTotalCubagemProdutosD365,
    t.tot_vida AS vlTotalCubagemProdutosVida
FROM spine s
LEFT JOIN media m ON m.seller_id = s.seller_id
LEFT JOIN total t ON t.seller_id = s.seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — cubagem é ESTÁTICA: D28/D56/D365/Vida dariam o MESMO valor por produto -----------------------
-- A cubagem não muda entre janelas; a janela só mudaria QUAIS SKUs entram.
SELECT oi.product_id,
       COUNT(DISTINCT p.product_length_cm * p.product_height_cm * p.product_width_cm) AS valores_distintos_de_cubagem
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY oi.product_id
HAVING valores_distintos_de_cubagem > 1   -- esperado: 0 linhas
LIMIT 20;

-- ----------------------- Prova B — média usa produto distinto; total usa unidade (grãos diferentes) -----------------------
SELECT oi.seller_id, oi.product_id,
       (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cubagem_cm3,
       COUNT(*) AS unidades_vendidas
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id, oi.product_id, cubagem_cm3
HAVING COUNT(*) > 1
ORDER BY unidades_vendidas DESC
LIMIT 20;

-- ----------------------- Prova C — qualquer dimensão NULL zera a cubagem (vira NULL, ignorada) -----------------------
SELECT COUNT(*) AS itens_vendidos,
       SUM(CASE WHEN p.product_length_cm IS NULL
                 OR p.product_height_cm IS NULL
                 OR p.product_width_cm  IS NULL THEN 1 ELSE 0 END) AS itens_dim_incompleta
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}';
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 9. `vlPrecoKg`  `*`

`SUM(price) / SUM(kg)` por janela (R$/kg). **Valor agregado** praticado. Inclui `vlPrecoKgAjustado` = `log1p` (extra, contém outliers).

In [ ]:
# Feature 09_vlPrecoKg  (fonte: features_sqlite/09_vlPrecoKg.sql)
SQL = """
-- =====================================================================
-- vlPrecoKg  (variaveis.md §6) — Preço por kg (R$/kg)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlPrecoKg{D28,D56,D365,Vida} (cru) + vlPrecoKgAjustado{W} (log1p)
-- Definição: receita total / massa total no período = SUM(price)/SUM(kg).
--   `...Ajustado` = ln(1+x) (log1p): produtos leves geram R$/kg enormes
--   (cauda assimétrica, skew ~30+; máx ~R$81 mil/kg); log1p comprime p/ ~0,45
--   de assimetria. Útil p/ modelos lineares/distância; mantemos a coluna crua
--   p/ modelos de árvore. (EXTRA — revisão externa, crítica 9.)
-- ---------------------------------------------------------------------
-- GRÃO: é uma RAZÃO entre dois totais da OPERAÇÃO (receita e massa). Ambos
--   são por UNIDADE vendida (não DISTINCT): cada venda gera receita e
--   embarca massa. Numerador e denominador usam a MESMA base (itens com
--   peso não-nulo), p/ o quociente ser coerente.
-- Premissas: receita = price (NÃO inclui frete — ver vlFreteKg); price é por
--   unidade (grão da fato) -> SUM(price) = receita total; denominador
--   0/NULL -> NULL (NULLIF).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens com peso não-nulo (base comum de num e den), por unidade.
WITH vendas AS (
    SELECT oi.seller_id,
           oi.price                   AS price,
           p.product_weight_g         AS w,
           o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
      AND p.product_weight_g IS NOT NULL
),
-- 2) componentes da razão por janela: receita (R$) e massa (kg) lado a lado.
componentes AS (
    SELECT seller_id,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN price END)      AS receita_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN w END)/1000.0    AS kg_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN price END)      AS receita_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN w END)/1000.0    AS kg_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN price END)      AS receita_d365,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN w END)/1000.0    AS kg_d365,
        SUM(price)                                                                           AS receita_vida,
        SUM(w)/1000.0                                                                        AS kg_vida
    FROM vendas
    GROUP BY seller_id
),
-- 3) razão receita/kg por janela; NULLIF protege denominador 0/NULL.
razoes AS (
    SELECT seller_id,
        receita_d28  / NULLIF(kg_d28,  0) AS pk_d28,
        receita_d56  / NULLIF(kg_d56,  0) AS pk_d56,
        receita_d365 / NULLIF(kg_d365, 0) AS pk_d365,
        receita_vida / NULLIF(kg_vida, 0) AS pk_vida
    FROM componentes
)
-- 4) coluna crua + versão Ajustado = ln(1+x) (log1p). NULL -> NULL.
SELECT
    seller_id,
    pk_d28  AS vlPrecoKgD28,
    pk_d56  AS vlPrecoKgD56,
    pk_d365 AS vlPrecoKgD365,
    pk_vida AS vlPrecoKgVida,
    ln(1 + pk_d28)  AS vlPrecoKgAjustadoD28,
    ln(1 + pk_d56)  AS vlPrecoKgAjustadoD56,
    ln(1 + pk_d365) AS vlPrecoKgAjustadoD365,
    ln(1 + pk_vida) AS vlPrecoKgAjustadoVida
FROM razoes;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — itens com peso NULL saem do NUM e do DEN (mesma base) -----------------------
-- Sellers onde receita_todos != receita_com_peso são os afetados pela exclusão.
SELECT oi.seller_id,
       SUM(oi.price)                                                  AS receita_todos_itens,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN oi.price END) AS receita_itens_com_peso
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id
HAVING SUM(CASE WHEN p.product_weight_g IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY receita_todos_itens DESC
LIMIT 20;

-- ----------------------- Prova B — componentes da razão + NULLIF (Vida): num R$, den kg, R$/kg -----------------------
SELECT oi.seller_id,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN oi.price END)                 AS num_receita_rs,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN p.product_weight_g END)/1000.0 AS den_kg,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN oi.price END)
         / NULLIF(SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN p.product_weight_g END)/1000.0, 0) AS preco_kg
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id
ORDER BY preco_kg DESC
LIMIT 20;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 10. `vlFreteKg`  `*`

`SUM(freight_value) / SUM(kg)` por janela (R$/kg). **Custo logístico**. Inclui `vlFreteKgAjustado` = `log1p` (extra).

In [ ]:
# Feature 10_vlFreteKg  (fonte: features_sqlite/10_vlFreteKg.sql)
SQL = """
-- =====================================================================
-- vlFreteKg  (variaveis.md §6) — Frete por kg (R$/kg)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlFreteKg{D28,D56,D365,Vida} (cru) + vlFreteKgAjustado{W} (log1p)
-- Definição: frete total / massa total no período = SUM(freight_value)/SUM(kg).
--   `...Ajustado` = ln(1+x) (log1p): mesma cauda assimétrica do preço/kg
--   (skew ~30+; máx ~R$17 mil/kg). Mantemos a coluna crua. (EXTRA — crítica 9.)
-- ---------------------------------------------------------------------
-- GRÃO: razão entre dois totais por UNIDADE vendida (não DISTINCT). Mesma
--   base (itens com peso não-nulo) no num e no den. freight_value já vem
--   ALOCADO por item pela Olist — não fazemos rateio. Denominador 0/NULL ->
--   NULL (NULLIF). Idêntico ao vlPrecoKg, trocando price por freight_value.
-- Data     : order_purchase_timestamp < {data_corte}.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens com peso não-nulo (base comum), por unidade.
WITH vendas AS (
    SELECT oi.seller_id,
           oi.freight_value           AS frete,
           p.product_weight_g         AS w,
           o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
      AND p.product_weight_g IS NOT NULL
),
-- 2) componentes da razão por janela: frete (R$) e massa (kg).
componentes AS (
    SELECT seller_id,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN frete END)      AS frete_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN w END)/1000.0    AS kg_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN frete END)      AS frete_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN w END)/1000.0    AS kg_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN frete END)      AS frete_d365,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN w END)/1000.0    AS kg_d365,
        SUM(frete)                                                                           AS frete_vida,
        SUM(w)/1000.0                                                                        AS kg_vida
    FROM vendas
    GROUP BY seller_id
),
-- 3) razão frete/kg por janela; NULLIF protege denominador 0/NULL.
razoes AS (
    SELECT seller_id,
        frete_d28  / NULLIF(kg_d28,  0) AS fk_d28,
        frete_d56  / NULLIF(kg_d56,  0) AS fk_d56,
        frete_d365 / NULLIF(kg_d365, 0) AS fk_d365,
        frete_vida / NULLIF(kg_vida, 0) AS fk_vida
    FROM componentes
)
-- 4) coluna crua + versão Ajustado = ln(1+x) (log1p). NULL -> NULL.
SELECT
    seller_id,
    fk_d28  AS vlFreteKgD28,
    fk_d56  AS vlFreteKgD56,
    fk_d365 AS vlFreteKgD365,
    fk_vida AS vlFreteKgVida,
    ln(1 + fk_d28)  AS vlFreteKgAjustadoD28,
    ln(1 + fk_d56)  AS vlFreteKgAjustadoD56,
    ln(1 + fk_d365) AS vlFreteKgAjustadoD365,
    ln(1 + fk_vida) AS vlFreteKgAjustadoVida
FROM razoes;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — freight_value já vem por item (não rateamos) -----------------------
-- Mostra frete e peso por item: a soma é direta, sem distribuir frete de pedido.
SELECT oi.order_id, oi.seller_id, oi.freight_value, p.product_weight_g
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
ORDER BY oi.freight_value DESC
LIMIT 20;

-- ----------------------- Prova B — componentes da razão + NULLIF (Vida): num R$, den kg, R$/kg -----------------------
SELECT oi.seller_id,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN oi.freight_value END)         AS num_frete_rs,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN p.product_weight_g END)/1000.0 AS den_kg,
       SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN oi.freight_value END)
         / NULLIF(SUM(CASE WHEN p.product_weight_g IS NOT NULL THEN p.product_weight_g END)/1000.0, 0) AS frete_kg
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_purchase_timestamp < '{data_corte}'
GROUP BY oi.seller_id
ORDER BY frete_kg DESC
LIMIT 20;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 11. `descTopCategoria{1,2,3}`  `*`

As **3 categorias mais vendidas** (por **quantidade vendida** = unidades) do seller, por janela. Posições inexistentes ficam `NULL`.

In [ ]:
# Feature 11_descTopCategoria  (fonte: features_sqlite/11_descTopCategoria.sql)
SQL = """
-- =====================================================================
-- descTopCategoria{1,2,3}  (variaveis.md §7) — Top 3 categorias do seller
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : descTopCategoria{1,2,3}{D28,D56,D365,Vida}  (12 colunas)
-- Definição: nome da 1ª/2ª/3ª categoria MAIS VENDIDA do seller na janela.
-- ---------------------------------------------------------------------
-- CRITÉRIO (variaveis.md §7): ranking por QUANTIDADE VENDIDA = nº de itens
--   vendidos (linhas de order_items) na categoria. Desempate determinístico:
--   unidades DESC -> pedidos distintos DESC -> categoria ASC. Usamos
--   ROW_NUMBER (não RANK) p/ garantir 1 categoria por posição.
-- Grão     : entidade = SELLER; agregamos por (seller, categoria) e a
--   "unidade vendida" é a linha de venda (cada item conta).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Nulos    : categoria NULL -> 'sem_categoria' (pode figurar no top); seller
--   com <k categorias na janela -> posição k NULL (guard u_>0).
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens vendidos até o corte, com categoria, pedido e data.
WITH vendas AS (
    SELECT oi.seller_id,
           COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
           oi.order_id,
           o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders       o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) por (seller, categoria): unidades vendidas e pedidos distintos por janela.
cat_base AS (
    SELECT seller_id, categoria,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN 1 ELSE 0 END)        AS u_d28,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN order_id END) AS p_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN 1 ELSE 0 END)        AS u_d56,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN order_id END) AS p_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN 1 ELSE 0 END)        AS u_d365,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN order_id END) AS p_d365,
        COUNT(*)                                                                                 AS u_vida,
        COUNT(DISTINCT order_id)                                                                 AS p_vida
    FROM vendas
    GROUP BY seller_id, categoria
),
-- 3) ranking por janela (unidades DESC -> pedidos DESC -> categoria ASC).
rk AS (
    SELECT seller_id, categoria, u_d28, u_d56, u_d365, u_vida,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d28  DESC, p_d28  DESC, categoria ASC) AS rk_d28,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d56  DESC, p_d56  DESC, categoria ASC) AS rk_d56,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d365 DESC, p_d365 DESC, categoria ASC) AS rk_d365,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_vida DESC, p_vida DESC, categoria ASC) AS rk_vida
    FROM cat_base
)
-- 4) pivota nome por posição × janela. "u_>0" evita rotular janela sem venda.
SELECT
    seller_id,
    MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN categoria END) AS descTopCategoria1D28,
    MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN categoria END) AS descTopCategoria1D56,
    MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN categoria END) AS descTopCategoria1D365,
    MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN categoria END) AS descTopCategoria1Vida,
    MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN categoria END) AS descTopCategoria2D28,
    MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN categoria END) AS descTopCategoria2D56,
    MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN categoria END) AS descTopCategoria2D365,
    MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN categoria END) AS descTopCategoria2Vida,
    MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN categoria END) AS descTopCategoria3D28,
    MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN categoria END) AS descTopCategoria3D56,
    MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN categoria END) AS descTopCategoria3D365,
    MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN categoria END) AS descTopCategoria3Vida
FROM rk
GROUP BY seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — ranking COMPLETO por unidades (Vida), com desempate -----------------------
-- Mostra unidades, pedidos e o ROW_NUMBER antes do pivot top1/2/3.
WITH vendas AS (
    SELECT oi.seller_id, COALESCE(p.product_category_name,'sem_categoria') AS categoria, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
cat AS (
    SELECT seller_id, categoria, COUNT(*) AS unidades, COUNT(DISTINCT order_id) AS pedidos
    FROM vendas GROUP BY seller_id, categoria
)
SELECT seller_id, categoria, unidades, pedidos,
       ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY unidades DESC, pedidos DESC, categoria ASC) AS rk
FROM cat
ORDER BY seller_id, rk
LIMIT 50;

-- ----------------------- Prova B — unidades (linhas) vs pedidos distintos divergem? (justifica COUNT(*) p/ "vendido") -----------------------
-- Mesma categoria pode ter vários itens no mesmo pedido: unidades > pedidos.
WITH vendas AS (
    SELECT oi.seller_id, COALESCE(p.product_category_name,'sem_categoria') AS categoria, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT seller_id, categoria, COUNT(*) AS unidades, COUNT(DISTINCT order_id) AS pedidos
FROM vendas
GROUP BY seller_id, categoria
HAVING COUNT(*) > COUNT(DISTINCT order_id)
ORDER BY unidades DESC
LIMIT 20;

-- ----------------------- Prova C — distribuição do nº de categorias por seller (<3 -> top2/top3 NULL) -----------------------
WITH cat_por_seller AS (
    SELECT oi.seller_id,
           COUNT(DISTINCT COALESCE(p.product_category_name,'sem_categoria')) AS n_categorias
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
    GROUP BY oi.seller_id
)
SELECT n_categorias, COUNT(*) AS qtd_sellers
FROM cat_por_seller
GROUP BY n_categorias
ORDER BY n_categorias;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 12. `vlShareTopCategoria{1,2,3}`  `*`

**Fração das unidades** concentrada nas 3 maiores categorias, por janela. Mede **concentração** (fragilidade).

In [ ]:
# Feature 12_vlShareTopCategoria  (fonte: features_sqlite/12_vlShareTopCategoria.sql)
SQL = """
-- =====================================================================
-- vlShareTopCategoria{1,2,3}  (variaveis.md §7) — Share das top 3 categorias
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlShareTopCategoria{1,2,3}{D28,D56,D365,Vida}  (12 colunas)
-- Definição: fração das UNIDADES vendidas concentrada em cada uma das 3
--            categorias mais vendidas (mesmo ranking de descTopCategoria).
-- ---------------------------------------------------------------------
-- CRITÉRIO: share = unidades(topk, janela) / unidades_totais(janela). Mesmo
--   ranking/desempate do §7 (unidades DESC -> pedidos DESC -> categoria ASC).
--   soma dos 3 shares <= 1. *1.0 força divisão real; total 0 -> NULL (NULLIF).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Nulos    : seller com <k categorias na janela -> share_k NULL.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) vendas: itens vendidos até o corte (categoria, pedido, data).
WITH vendas AS (
    SELECT oi.seller_id,
           COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
           oi.order_id,
           o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders       o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- 2) por (seller, categoria): unidades e pedidos distintos por janela.
cat_base AS (
    SELECT seller_id, categoria,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN 1 ELSE 0 END)        AS u_d28,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-28 days')  THEN order_id END) AS p_d28,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN 1 ELSE 0 END)        AS u_d56,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-56 days')  THEN order_id END) AS p_d56,
        SUM(CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN 1 ELSE 0 END)        AS u_d365,
        COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}','-365 days') THEN order_id END) AS p_d365,
        COUNT(*)                                                                                 AS u_vida,
        COUNT(DISTINCT order_id)                                                                 AS p_vida
    FROM vendas
    GROUP BY seller_id, categoria
),
-- 3) ranking por janela (idêntico ao descTopCategoria).
rk AS (
    SELECT seller_id, categoria, u_d28, u_d56, u_d365, u_vida,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d28  DESC, p_d28  DESC, categoria ASC) AS rk_d28,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d56  DESC, p_d56  DESC, categoria ASC) AS rk_d56,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d365 DESC, p_d365 DESC, categoria ASC) AS rk_d365,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_vida DESC, p_vida DESC, categoria ASC) AS rk_vida
    FROM cat_base
)
-- 4) share = unidades da posição k / total de unidades da janela (SUM por seller).
--    Guard "u_>0": se a posição k não existe na janela (seller com <k
--    categorias vendidas), o numerador vira NULL -> share NULL (não 0).
SELECT
    seller_id,
    MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN u_d28  END) * 1.0 / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria1D28,
    MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN u_d56  END) * 1.0 / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria1D56,
    MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN u_d365 END) * 1.0 / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria1D365,
    MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN u_vida END) * 1.0 / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria1Vida,
    MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN u_d28  END) * 1.0 / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria2D28,
    MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN u_d56  END) * 1.0 / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria2D56,
    MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN u_d365 END) * 1.0 / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria2D365,
    MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN u_vida END) * 1.0 / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria2Vida,
    MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN u_d28  END) * 1.0 / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria3D28,
    MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN u_d56  END) * 1.0 / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria3D56,
    MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN u_d365 END) * 1.0 / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria3D365,
    MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN u_vida END) * 1.0 / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria3Vida
FROM rk
GROUP BY seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — componentes do share (Vida): unidades por posição, total e fração -----------------------
WITH vendas AS (
    SELECT oi.seller_id, COALESCE(p.product_category_name,'sem_categoria') AS categoria, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
r AS (
    SELECT seller_id, categoria, COUNT(*) AS unidades,
           ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY COUNT(*) DESC, COUNT(DISTINCT order_id) DESC, categoria ASC) AS rk,
           SUM(COUNT(*)) OVER (PARTITION BY seller_id) AS unidades_totais
    FROM vendas GROUP BY seller_id, categoria
)
SELECT seller_id, rk, categoria, unidades, unidades_totais,
       unidades * 1.0 / NULLIF(unidades_totais, 0) AS share
FROM r
WHERE rk <= 3
ORDER BY seller_id, rk
LIMIT 50;

-- ----------------------- Prova B — soma dos 3 shares <= 1 (Vida): nenhuma violação esperada -----------------------
WITH vendas AS (
    SELECT oi.seller_id, COALESCE(p.product_category_name,'sem_categoria') AS categoria, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
r AS (
    SELECT seller_id, categoria, COUNT(*) AS unidades,
           ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY COUNT(*) DESC, COUNT(DISTINCT order_id) DESC, categoria ASC) AS rk,
           SUM(COUNT(*)) OVER (PARTITION BY seller_id) AS unidades_totais
    FROM vendas GROUP BY seller_id, categoria
)
SELECT COUNT(*) AS violacoes_soma_maior_que_1
FROM (
    SELECT seller_id, SUM(unidades) * 1.0 / NULLIF(MAX(unidades_totais), 0) AS soma_top3
    FROM r WHERE rk <= 3 GROUP BY seller_id
)
WHERE soma_top3 > 1.0000001;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 13. `vlShareProdutosSemCadastro`  (extra — Vida)

**Missingness de cadastro**: fração dos produtos distintos do seller sem categoria/descrição/foto/peso. Captura ausência que a média não vê. ⚠ categoria≡descrição≡foto neste dataset; `SemPeso` ~constante.

In [ ]:
# Feature 13_vlShareProdutosSemCadastro  (fonte: features_sqlite/13_vlShareProdutosSemCadastro.sql)
SQL = """
-- =====================================================================
-- vlShareProdutosSemCadastro  (EXTRA — revisão externa, crítica 7)
-- Janela: Vida (sem sufixo)        |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : vlShareProdutosSemCategoria, vlShareProdutosSemDescricao,
--            vlShareProdutosSemFoto, vlShareProdutosSemPeso  (frações [0,1])
-- Definição: fração dos PRODUTOS DISTINTOS do seller com atributo de
--   cadastro AUSENTE (NULL). É uma feature de **missingness** — captura
--   ausência de cadastro, que a média (AVG) NÃO vê (AVG descarta NULL; em
--   `product_photos_qty` não existe valor 0, só NULL → a ausência some da
--   média de fotos/descrição). Sinal de qualidade/engajamento do seller.
-- ---------------------------------------------------------------------
-- GRÃO: atributo do PRODUTO -> PRODUTOS DISTINTOS por seller (DISTINCT
--   product_id); denominador = nº de produtos distintos do seller.
-- Data     : order_purchase_timestamp < {data_corte}.
-- ⚠ Achados no Olist (corte 2018-07-01): os 580 produtos sem categoria são
--   EXATAMENTE os mesmos sem descrição e sem foto -> as 3 colunas de
--   cadastro são colineares (idênticas) NESTE dataset; mantidas separadas
--   porque a equivalência pode quebrar num refresh. `SemPeso` é quase
--   constante (só 2 produtos sem peso) -> candidata a descarte no modelo.
-- Parâmetro: {data_corte} (ex. 2018-07-01).
-- =====================================================================

-- 1) produtos: PRODUTOS DISTINTOS vendidos pelo seller (cada SKU 1x), com os
--    atributos de cadastro.
WITH produtos AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.product_id,
        p.product_category_name    AS categoria,
        p.product_description_lenght AS descricao,
        p.product_photos_qty       AS fotos,
        p.product_weight_g         AS peso
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
-- 2) share de ausência por atributo = média do indicador (NULL -> 1) sobre os
--    produtos distintos. Denominador (COUNT) > 0 sempre (seller tem ≥1 venda).
SELECT
    seller_id,
    AVG(CASE WHEN categoria IS NULL THEN 1.0 ELSE 0 END) AS vlShareProdutosSemCategoria,
    AVG(CASE WHEN descricao IS NULL THEN 1.0 ELSE 0 END) AS vlShareProdutosSemDescricao,
    AVG(CASE WHEN fotos     IS NULL THEN 1.0 ELSE 0 END) AS vlShareProdutosSemFoto,
    AVG(CASE WHEN peso      IS NULL THEN 1.0 ELSE 0 END) AS vlShareProdutosSemPeso
FROM produtos
GROUP BY seller_id;

-- ====================== ANÁLISE / PROVAS DA DECISÃO (rodar manualmente) ======================
-- A feature é o bloco 1; as provas começam no bloco 2. Rode com:
--   python scripts/run_feature_sqlite.py <este_arquivo> --list
--   python scripts/run_feature_sqlite.py <este_arquivo> --block 2
-- =============================================================================================

-- ----------------------- Prova A — missingness é INVISÍVEL à média (fotos NULL, não 0) -----------------------
-- product_photos_qty nunca é 0 (mín=1); produtos sem cadastro têm NULL, que o
-- AVG de vlMediaFotosProduto descarta. Logo a média não enxerga a ausência.
WITH produtos AS (
    SELECT DISTINCT oi.product_id, p.product_photos_qty AS fotos
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT MIN(fotos) AS min_fotos_nao_nulo,
       SUM(CASE WHEN fotos = 0 THEN 1 ELSE 0 END)    AS produtos_fotos_zero,
       SUM(CASE WHEN fotos IS NULL THEN 1 ELSE 0 END) AS produtos_fotos_null
FROM produtos;

-- ----------------------- Prova B — categoria/descrição/foto AUSENTES são os MESMOS produtos (colinearidade) -----------------------
-- Esperado: os três NULL coincidem (cadastro vazio em bloco) -> 3 shares idênticos.
WITH produtos AS (
    SELECT DISTINCT oi.product_id,
           p.product_category_name AS cat, p.product_description_lenght AS descr, p.product_photos_qty AS fotos
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    SUM(CASE WHEN cat IS NULL THEN 1 ELSE 0 END)   AS sem_categoria,
    SUM(CASE WHEN descr IS NULL THEN 1 ELSE 0 END) AS sem_descricao,
    SUM(CASE WHEN fotos IS NULL THEN 1 ELSE 0 END) AS sem_fotos,
    SUM(CASE WHEN cat IS NULL AND descr IS NULL AND fotos IS NULL THEN 1 ELSE 0 END) AS sem_os_tres,
    SUM(CASE WHEN cat IS NULL AND descr IS NOT NULL THEN 1 ELSE 0 END)               AS divergencia_cat_desc
FROM produtos;

-- ----------------------- Prova C — distribuição do share (quantos sellers têm catálogo todo sem cadastro) -----------------------
-- Esperado: ~250 sellers com share>0 e ~60 com share==1 (catálogo inteiro sem cadastro).
WITH produtos AS (
    SELECT DISTINCT oi.seller_id, oi.product_id, p.product_category_name AS cat
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
por_seller AS (
    SELECT seller_id, AVG(CASE WHEN cat IS NULL THEN 1.0 ELSE 0 END) AS share
    FROM produtos GROUP BY seller_id
)
SELECT
    SUM(CASE WHEN share > 0      THEN 1 ELSE 0 END) AS sellers_com_algum_sem_cadastro,
    SUM(CASE WHEN share >= 0.999 THEN 1 ELSE 0 END) AS sellers_catalogo_todo_sem_cadastro,
    COUNT(*)                                        AS total_sellers
FROM por_seller;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

---

### Pronto

Cada tabela acima é o resultado da variável para todos os sellers ativos no corte. Para a tabela larga única, basta um `LEFT JOIN` dos 13 resultados por `seller_id`.